# Task 0: Data Audit and Exploratory Data Analysis

## 1. Introduction

Audit the metadata and images, inspect label imbalance and examples, then validate
the shared split and training-only normalization. The notebook presents the
decisions; the code cells below contain the audit
and grouping steps. Existing split/normalization files are read and validated,
never silently repaired or rewritten. Creation is only for a missing manifest.

## 2. Library Imports & Setup


In [ ]:
from IPython.display import display
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / 'scripts'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image, UnidentifiedImageError
from sklearn.metrics import normalized_mutual_info_score

from scripts.preprocessing import (
    IMAGE_AUDIT_PATH, IMAGE_SIZE, NORMALISATION_PATH, OUTPUT_DIR, SEED, SPLIT_PATH, TARGETS,
    load_metadata, load_prediction_template, preprocess_image,
)
sns.set_theme(style='whitegrid')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
from scripts.preprocessing import file_sha256

from sklearn.model_selection import GroupShuffleSplit


## 3. Load Metadata

Inspect the catalogue, image coverage and label fields before examining distributions. Literal `NA` is retained as a catalogue value; empty fields remain missing.

In [ ]:
metadata = load_metadata()
template = load_prediction_template()
train_image_dir = Path(metadata.image_path.iloc[0]).parent
test_image_dir = Path(template.image_path.iloc[0]).parent
train_image_ids = {path.stem for path in train_image_dir.glob('*.jpg')}
test_image_ids = {path.stem for path in test_image_dir.glob('*.jpg')}
repaired_name_mask = metadata.get(
    'productDisplayName_repaired', pd.Series(False, index=metadata.index)
).astype(bool)
coverage_report = {
    'training_rows': len(metadata),
    'unique_training_ids': int(metadata.id.nunique()),
    'duplicate_training_ids': int(metadata.id.duplicated().sum()),
    'training_images_found': int(metadata.has_image.sum()),
    'missing_training_ids': metadata.loc[~metadata.has_image, 'id'].tolist(),
    'orphan_training_image_ids': sorted(train_image_ids - set(metadata.id)),
    'test_rows': len(template),
    'unique_test_ids': int(template.id.nunique()),
    'missing_test_ids': template.loc[~template.image_path.map(lambda value: Path(value).is_file()), 'id'].tolist(),
    'orphan_test_image_ids': sorted(test_image_ids - set(template.id)),
    'styles_train_sha256': file_sha256(train_image_dir.parent / 'styles_train.csv'),
    'styles_prediction_sha256': file_sha256(test_image_dir.parent / 'styles_prediction.csv'),
    'blank_labels': {target: int(metadata[target].str.strip().eq('').sum()) for target in TARGETS},
    'repaired_product_display_names': int(repaired_name_mask.sum()),
    'repaired_product_display_name_ids': metadata.loc[repaired_name_mask, 'id'].tolist(),
}
coverage_report

## 4. Exploratory Data Analysis (EDA)

Inspect image validity, duplicate groups, class support, relationships and visual examples. Validate the frozen split and training-only normalization before model development.

### 4.1. Image Validity & Duplicate Audit

Decode images and compute content hashes to detect corrupt files and duplicates. Use the displayed audit counts in the final analysis.

In [ ]:
def load_image_audit(metadata, force=False):
    available = metadata.loc[metadata.has_image]
    if force or not IMAGE_AUDIT_PATH.exists():
        rows = []
        for row in available.itertuples(index=False):
            result = dict(id=row.id, sha256='', width='', height='', mode='', decode_error='')
            try:
                digest = file_sha256(row.image_path).lower()
                with Image.open(row.image_path) as image:
                    image.load()
                    result.update(sha256=digest, width=image.width, height=image.height, mode=image.mode)
            except (OSError, UnidentifiedImageError) as error:
                result['decode_error'] = type(error).__name__
            rows.append(result)
        IMAGE_AUDIT_PATH.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(rows).to_csv(IMAGE_AUDIT_PATH, index=False)
    audit = pd.read_csv(IMAGE_AUDIT_PATH, dtype={'id': 'string'}, keep_default_na=False)
    if audit.id.duplicated().any() or set(audit.id) != set(available.id):
        raise ValueError('Cached image audit does not match available IDs; repeat the full audit')
    if (audit.decode_error.eq('') & audit.sha256.eq('')).any():
        raise ValueError('Decoded audit rows must have a SHA-256 hash')
    return audit


In [ ]:
audit_setting = os.environ.get('FASHION_RUN_FULL_AUDIT', 'auto').strip().lower()
force_audit = audit_setting in {'1', 'true', 'yes'}
image_audit = load_image_audit(metadata, force=force_audit)
print(f'Audit rows: {len(image_audit):,}; forced rescan: {force_audit}')

In [ ]:
usable_audit = image_audit.loc[image_audit.decode_error.eq('') & image_audit.sha256.ne('')]
usable_metadata = metadata.merge(usable_audit[['id', 'sha256']], on='id', validate='one_to_one')
hash_counts = usable_audit.sha256.value_counts()
duplicate_hashes = hash_counts.loc[hash_counts.gt(1)].index
duplicate_rows = usable_metadata.loc[usable_metadata.sha256.isin(duplicate_hashes)]
conflict_rows = []
for target in TARGETS:
    counts = duplicate_rows.groupby('sha256')[target].nunique()
    conflicts = counts.loc[counts.gt(1)].index
    conflict_rows.append({
        'target': target,
        'conflicting_hash_groups': len(conflicts),
        'rows_in_conflicting_groups': int(duplicate_rows.sha256.isin(conflicts).sum()),
    })
duplicate_conflict_summary = pd.DataFrame(conflict_rows).set_index('target')
lost_labels = {}
for target in TARGETS:
    original = set(metadata.loc[metadata[target].str.strip().ne(''), target])
    lost_labels[target] = sorted(original - set(usable_metadata[target]))
audit_report = {
    **coverage_report,
    'decoded_images': int(image_audit.decode_error.eq('').sum()),
    'decode_errors': image_audit.loc[image_audit.decode_error.ne(''), 'id'].tolist(),
    'exact_duplicate_hash_groups': len(duplicate_hashes),
    'rows_in_exact_duplicate_groups': len(duplicate_rows),
    'labels_lost_after_image_cleaning': lost_labels,
    'image_modes': image_audit['mode'].value_counts().to_dict(),
}
report_path = OUTPUT_DIR / 'audit.json'
if not report_path.exists() or json.loads(report_path.read_text()) != audit_report:
    report_path.write_text(json.dumps(audit_report, indent=2), encoding='utf-8')
display(audit_report)
display(duplicate_conflict_summary)

#### Audit Observations

**Complete after execution:** Report missing/corrupt images, duplicate-group counts and conflicting labels from the audit tables. Explain why duplicate groups must stay together across partitions.

### 4.2. Target Distributions & Missing Labels

Compare class support, missing values and the share of the largest class. This motivates reporting macro-F1 alongside accuracy.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 14))
for target, axis in zip(TARGETS, axes.ravel()):
    counts = usable_metadata[target].replace('', '<blank>').value_counts().head(30)
    sns.barplot(x=counts.values, y=counts.index, ax=axis)
    axis.set_xscale('log')
    axis.set_title(f'{target}: largest classes')
    axis.set_xlabel('Images (log scale)')
    axis.bar_label(axis.containers[0], fmt='%d', padding=3, fontsize=7)
plt.tight_layout()
figures = ROOT / 'figures'
figures.mkdir(exist_ok=True)
fig.savefig(figures / 'target_distributions.png', dpi=220, bbox_inches='tight')
distribution_rows = []
rare_counts = {}
for target in TARGETS:
    values = usable_metadata[target]
    counts = values.loc[values.str.strip().ne('')].value_counts()
    distribution_rows.append({
        'target': target, 'classes': len(counts),
        'blank': int(values.str.strip().eq('').sum()),
        'largest_class_share': counts.iloc[0] / counts.sum(),
        'classes_below_10': int(counts.lt(10).sum()),
    })
    rare_counts[target] = counts.sort_values().head(15).rename('images')
display(pd.DataFrame(distribution_rows).set_index('target'))
display(pd.concat(rare_counts, names=['target', 'label']).to_frame())

#### Distribution Observations

**Complete after execution:** Identify dominant and rare labels for each target using the generated tables. Do not treat strong overall accuracy as evidence of good performance on minority labels.

### 4.3. Target Relationships & Confounding

Cross-tabulations and normalized mutual information indicate relationships between labels. These associations can support prediction but can also encourage shortcuts.

In [ ]:
relationship_summary = pd.DataFrame([
    {
        'relationship': f'articleType vs {target}',
        'normalized_mutual_information': normalized_mutual_info_score(
            usable_metadata.loc[usable_metadata[target].str.strip().ne(''), 'articleType'],
            usable_metadata.loc[usable_metadata[target].str.strip().ne(''), target],
        ),
    }
    for target in ('season', 'gender', 'usage')
]).set_index('relationship')
season_by_type = pd.crosstab(usable_metadata.articleType, usable_metadata.season, normalize='index')
season_support = usable_metadata.articleType.value_counts()
concentrated_season_types = (season_by_type.assign(
    support=season_support, dominant_share=season_by_type.max(axis=1),
).loc[lambda frame: frame.support.ge(20)].sort_values('dominant_share', ascending=False).head(15))
display(relationship_summary)
concentrated_season_types.style.background_gradient(cmap='Blues')
pd.crosstab(usable_metadata.gender, usable_metadata.usage, normalize='index').style.background_gradient(cmap='Purples')

### 4.4. Visual Inspection

Inspect representative and problematic images alongside metadata. Describe ambiguity, backgrounds and other visual factors that may influence classification.

In [ ]:
missing_examples = metadata.loc[~metadata.has_image, ['id', 'articleType', 'season', 'gender', 'usage']].assign(issue='missing image')
decode_examples = image_audit.loc[image_audit.decode_error.ne(''), ['id', 'decode_error']].assign(issue='decode error')
display(missing_examples)
display(decode_examples)

grayscale_ids = image_audit.loc[image_audit['mode'].eq('L'), 'id'].head(4)
grayscale_examples = usable_metadata.loc[usable_metadata.id.isin(grayscale_ids)].assign(example_reason='grayscale')
conflicting_hashes = (
    duplicate_rows.groupby('sha256')[list(TARGETS)].nunique().gt(1).any(axis=1)
).loc[lambda values: values].index[:2]
conflict_examples = duplicate_rows.loc[duplicate_rows.sha256.isin(conflicting_hashes)].groupby('sha256').head(2).assign(example_reason='duplicate label conflict')
article_support = usable_metadata.articleType.value_counts()
rare_examples = usable_metadata.loc[usable_metadata.articleType.isin(article_support.loc[article_support.le(2)].index)].drop_duplicates('articleType').head(4).assign(example_reason='rare article type')
chosen_ids = set(pd.concat([grayscale_examples, conflict_examples, rare_examples]).id)
common_examples = usable_metadata.loc[~usable_metadata.id.isin(chosen_ids)].sample(4, random_state=SEED).assign(example_reason='random catalogue sample')
sample = pd.concat([grayscale_examples, conflict_examples, rare_examples, common_examples], ignore_index=True).drop_duplicates('id').head(16)
fig, axes = plt.subplots(4, 4, figsize=(11, 13))
for (_, row), axis in zip(sample.iterrows(), axes.ravel()):
    with Image.open(row.image_path) as image:
        axis.imshow(image.convert('RGB'))
    axis.set_title(f'{row.example_reason}\n{row.articleType} | {row.season}', fontsize=8)
    axis.axis('off')
for axis in axes.ravel()[len(sample):]:
    axis.axis('off')
plt.tight_layout()
dimensions = image_audit.loc[image_audit.decode_error.eq(''), ['width', 'height']].apply(pd.to_numeric)
display(dimensions.describe())
display(image_audit.loc[image_audit.decode_error.eq(''), 'mode'].value_counts())
display(dimensions.value_counts().rename('images').head(10).to_frame())

### 4.5. Group-Isolated Split Validation

The following functions group related rows, ensure label coverage and validate the frozen manifest. Existing valid assignments are reused; group isolation is more important than matching exact percentages.

#### Build Group Keys

In [ ]:
def build_group_keys(frame, audit):
    frame = frame.merge(audit[['id', 'sha256', 'decode_error']], on='id', how='left')
    frame = frame.loc[frame.has_image & frame.decode_error.eq('')].reset_index(drop=True)
    frame['name_key'] = frame.productDisplayName.astype('string').str.lower().str.replace('\\W+', ' ', regex=True).str.strip()
    frame.loc[frame.name_key.eq(''), 'name_key'] = frame.id
    parent = list(range(len(frame)))

    def find(index):
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = parent[index]
        return index

    def union(left, right):
        left_root, right_root = (find(left), find(right))
        if left_root != right_root:
            parent[right_root] = left_root
    for column in ('name_key', 'sha256'):
        values = frame[column].astype('string').fillna('')
        for value, indices in values.groupby(values, sort=False).groups.items():
            if not value or len(indices) < 2:
                continue
            first = int(indices[0])
            for index in indices[1:]:
                union(first, int(index))
    frame['group_key'] = [f'group_{find(index)}' for index in range(len(frame))]
    return frame


#### Enforce Training Label Coverage

In [ ]:
def enforce_training_label_coverage(frame, split_frame):
    """Move whole duplicate groups to train when a label has no train example."""
    revised = split_frame.copy()
    changes = []
    for target in TARGETS:
        joined = frame[['id', target]].merge(revised, on='id', validate='one_to_one')
        valid = joined[target].astype('string').str.strip().ne('')
        train_labels = set(joined.loc[valid & joined.split.eq('train'), target])
        missing_labels = sorted(set(joined.loc[valid, target]) - train_labels)
        for label in missing_labels:
            groups = joined.loc[valid & joined[target].eq(label), 'group_key'].unique()
            move_mask = revised.group_key.isin(groups) & ~revised.split.eq('train')
            changes.append({'target': target, 'label': label, 'groups_moved': int(len(groups)), 'rows_moved': int(move_mask.sum())})
            revised.loc[revised.group_key.isin(groups), 'split'] = 'train'
    return (revised, pd.DataFrame(changes))


#### Create Frozen Split

In [ ]:
def create_frozen_split(frame, audit):
    grouped = build_group_keys(frame, audit)
    first = GroupShuffleSplit(n_splits=1, train_size=0.7, random_state=SEED)
    _, holdout_index = next(first.split(grouped, groups=grouped.group_key))
    holdout = grouped.iloc[holdout_index]
    second = GroupShuffleSplit(n_splits=1, train_size=0.5, random_state=SEED + 1)
    _, test_relative = next(second.split(holdout, groups=holdout.group_key))
    grouped['split'] = 'train'
    grouped.loc[grouped.id.isin(holdout.id), 'split'] = 'validation'
    grouped.loc[grouped.id.isin(holdout.iloc[test_relative].id), 'split'] = 'test'
    return grouped[['id', 'group_key', 'split']]


#### Validate Frozen Split

In [ ]:
def validate_frozen_split(metadata, audit, splits):
    grouped = build_group_keys(metadata, audit)
    if splits.id.duplicated().any() or set(splits.id) != set(grouped.id):
        raise ValueError('Frozen split IDs differ from the usable audited images')
    if splits.group_key.isna().any() or splits.group_key.eq('').any():
        raise ValueError('Every split row needs a group key')
    if set(splits.split) != {'train', 'validation', 'test'}:
        raise ValueError('Expected nonempty train, validation and test partitions')
    if splits.groupby('group_key').split.nunique().gt(1).any():
        raise ValueError('A frozen group crosses partitions')
    # Recompute connections to catch a manifest that assigns different group keys
    # to rows sharing a name/hash, including transitive chains.
    joined = grouped[['id', 'group_key']].merge(splits[['id', 'split']], on='id', validate='one_to_one')
    if joined.groupby('group_key').split.nunique().gt(1).any():
        raise ValueError('Related product names or duplicate images cross partitions')
    _, changes = enforce_training_label_coverage(metadata, splits)
    if not changes.empty:
        raise ValueError('Frozen training split lacks evaluated labels; review the manifest explicitly')


#### Load Or Create Split

In [ ]:
def load_or_create_split(metadata, audit):
    if SPLIT_PATH.exists():
        splits = pd.read_csv(SPLIT_PATH, dtype={'id': 'string'})
        validate_frozen_split(metadata, audit, splits)
        return splits
    splits = create_frozen_split(metadata, audit)
    splits, _ = enforce_training_label_coverage(metadata, splits)
    validate_frozen_split(metadata, audit, splits)
    SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)
    splits.to_csv(SPLIT_PATH, index=False)
    return splits


In [ ]:
splits = load_or_create_split(metadata, image_audit)
print('Frozen split validated; existing assignments preserved.')
display(splits['split'].value_counts())
split_metadata = usable_metadata.merge(splits, on='id', validate='one_to_one')
split_counts = splits.split.value_counts().reindex(['train', 'validation', 'test'])
split_summary = pd.DataFrame({
    'rows': split_counts,
    'percent': 100 * split_counts / len(splits),
    'groups': splits.groupby('split').group_key.nunique().reindex(split_counts.index),
})
split_quality_rows = []
for target in TARGETS:
    valid = split_metadata[target].str.strip().ne('')
    target_frame = split_metadata.loc[valid]
    overall_share = target_frame[target].value_counts(normalize=True)
    train_labels = set(target_frame.loc[target_frame.split.eq('train'), target])
    for split_name in ('train', 'validation', 'test'):
        partition = target_frame.loc[target_frame.split.eq(split_name), target]
        partition_share = partition.value_counts(normalize=True).reindex(overall_share.index, fill_value=0)
        split_quality_rows.append({
            'target': target, 'split': split_name, 'usable_rows': len(partition),
            'labels_present': partition.nunique(),
            'labels_absent_from_training': len(set(partition) - train_labels),
            'max_class_share_drift_pp': 100 * (partition_share - overall_share).abs().max(),
        })
split_quality = pd.DataFrame(split_quality_rows)
display(split_summary.round({'percent': 2}))
display(pd.DataFrame({
    'groups': [splits.group_key.nunique()],
    'largest_group_rows': [splits.groupby('group_key').size().max()],
    'groups_crossing_splits': [splits.groupby('group_key').split.nunique().gt(1).sum()],
}))
split_quality.round({'max_class_share_drift_pp': 2})

#### Split Observations

**Complete after execution:** Report actual partition sizes, label coverage and distribution differences. Confirm that no group crosses partitions; disclose the internal test?s prior development exposure.

### 4.6. Training-Only RGB Normalization

Compute channel statistics from training images only, then reuse them unchanged for validation, test and application inference.

In [ ]:
def compute_training_normalisation(frame, split_frame):
    joined = frame.merge(split_frame, on='id', validate='one_to_one')
    paths = joined.loc[joined.split.eq('train'), 'image_path']
    channel_sum = np.zeros(3, dtype=np.float64)
    channel_square_sum = np.zeros(3, dtype=np.float64)
    pixel_count = 0
    for path in paths:
        with Image.open(path) as image:
            array = preprocess_image(image).astype(np.float64)
        channel_sum += array.sum(axis=(0, 1))
        channel_square_sum += np.square(array).sum(axis=(0, 1))
        pixel_count += array.shape[0] * array.shape[1]
    mean = channel_sum / pixel_count
    std = np.sqrt(channel_square_sum / pixel_count - np.square(mean))
    return {'mean': mean.tolist(), 'std': std.tolist(), 'image_size': list(IMAGE_SIZE)}


In [ ]:
def load_or_compute_normalisation(metadata, splits):
    if NORMALISATION_PATH.exists():
        normalisation = json.loads(NORMALISATION_PATH.read_text(encoding='utf-8'))
    else:
        normalisation = compute_training_normalisation(metadata, splits)
        NORMALISATION_PATH.write_text(json.dumps(normalisation, indent=2), encoding='utf-8')
    mean = np.asarray(normalisation['mean'])
    std = np.asarray(normalisation['std'])
    if (normalisation['image_size'] != list(IMAGE_SIZE) or mean.shape != (3,)
            or std.shape != (3,) or not np.isfinite(mean).all()
            or not np.isfinite(std).all() or (std <= 0).any()):
        raise ValueError('Invalid frozen RGB normalization')
    return normalisation


In [ ]:
normalisation = load_or_compute_normalisation(metadata, splits)
normalisation

## 5. Observations & Discussion

**Complete after execution:** Summarize measured metadata issues, image quality, duplicates, class imbalance and target relationships. Explain how the findings justify group-isolated partitions, training-only normalization, macro-F1 selection and later error analysis. Confirm that Tasks 1?4 use the exported manifests; do not invent counts before inspecting the outputs.